# 3. Analytical Feature Engineering Layer

## Feature Engineering Objective

After the raw source data was retrieved and converted into standardized annual records, the next requirement was to transform those records into variables that could support comparison, interpretation, and downstream delivery.

The annual snapshot layer provided a stable internal structure, but the raw fields inside each record were still not sufficient for analytical use. Revenue, net income, cash flow, equity, debt, and share-count values become more meaningful only after they are reorganized into features that describe growth, profitability, cash conversion, financial condition, and price-relative context.

For that reason, this stage was designed as an analytical feature engineering layer rather than as a simple ratio-calculation step. The purpose was to translate normalized annual records into reusable variables that could later support rule-based logic, application-layer review, and AI-assisted interpretation.

## Feature Design Strategy

The feature layer was designed around three principles.

First, the engineered variables had to reflect real analytical questions. Growth features were used to summarize long-term and short-term change. Margin features were used to relate profit and cash generation to scale. Cash-conversion features were used to compare reported earnings with realized cash. Safety features were used to capture liquidity and leverage. Valuation-related features were used to connect normalized company records with current market context.

Second, all engineered variables had to be computed from the standardized annual snapshot layer rather than directly from raw source responses. This preserved the separation between source handling and downstream analytical logic.

Third, the selected features had to remain interpretable. The goal was not to maximize the number of outputs, but to build a set of variables that could be reused consistently across tables, application sections, rule-based summaries, and AI-generated interpretation.

## Core Feature Groups

The engineered features were grouped into several analytical categories so that the workflow could move beyond raw reporting values and represent multiple dimensions of company behavior.

### Growth features
Growth features were used to summarize directional change across time.  
The workflow included compound annual growth rates and year-over-year growth rates for revenue, net income, free cash flow, earnings per share, and book value per share.

### Profitability and efficiency features
Profitability features were used to evaluate how effectively revenue was translated into earnings and free cash flow.  
The workflow included net margin, operating margin, and free-cash-flow margin.

### Cash-conversion features
Cash-conversion features were used to compare accounting earnings with actual cash generation.  
The workflow included operating-cash-flow-to-net-income and free-cash-flow-to-net-income relationships.

### Financial condition features
Financial-condition features were used to capture liquidity, leverage, and balance-sheet flexibility.  
The workflow included current ratio, debt-to-equity ratio, and net cash position.

### Valuation-related features
Valuation-related features were used to connect normalized company records with external price context.  
The workflow included EPS, book value per share, free cash flow per share, price multiples, and yield-based outputs.

In [ ]:
def compute_growth_metrics(snapshots: List[Any]) -> Optional[Dict[str, Any]]:
    snaps = _sorted_snaps(snapshots)
    if len(snaps) < 2:
        return None

    rev_cagr, rev_years = _cagr_from_snapshots(
        snaps, lambda s: getattr(s, "revenue", None)
    )
    ni_cagr, ni_years = _cagr_from_snapshots(
        snaps, lambda s: getattr(s, "net_income", None)
    )
    fcf_cagr, fcf_years = _cagr_from_snapshots(
        snaps, _compute_fcf
    )
    eps_cagr, eps_years = _cagr_from_snapshots(
        snaps, _compute_eps
    )
    bvps_cagr, bvps_years = _cagr_from_snapshots(
        snaps, _compute_bvps
    )

In [ ]:
def compute_yoy_growth(snapshots: List[Any]) -> List[Dict[str, Any]]:
    snaps = _sorted_snaps(snapshots)
    if len(snaps) < 2:
        return []

    def growth(prev_val: Optional[float], curr_val: Optional[float]) -> Optional[float]:
        if prev_val is None or curr_val is None or prev_val == 0:
            return None
        return (curr_val - prev_val) / prev_val

    results: List[Dict[str, Any]] = []

    for i in range(1, len(snaps)):
        prev = snaps[i - 1]
        curr = snaps[i]

        results.append({
            "year_from": prev.fiscal_year,
            "year_to": curr.fiscal_year,
            "rev_growth": growth(getattr(prev, "revenue", None), getattr(curr, "revenue", None)),
            "ni_growth": growth(getattr(prev, "net_income", None), getattr(curr, "net_income", None)),
            "fcf_growth": growth(_compute_fcf(prev), _compute_fcf(curr)),
            "eps_growth": growth(_compute_eps(prev), _compute_eps(curr)),
            "bvps_growth": growth(_compute_bvps(prev), _compute_bvps(curr)),
        })

    return results

In [ ]:
def _compute_fcf(snapshot) -> Optional[float]:
    ocf = getattr(snapshot, "operating_cash_flow", None)
    capex = getattr(snapshot, "capex", None)

    if ocf is None or capex is None:
        return None

    return ocf - capex

## Why This Layer Matters

This layer was the point at which the workflow became analytically meaningful.

The retrieval layer collected external records, and the schema layer standardized them into stable annual structures. The feature engineering layer then transformed those structures into variables that could support comparison across time, relationship analysis across financial dimensions, and integration with application-level review logic.

From a data perspective, this layer demonstrated that the project was not limited to storage or retrieval. It introduced reusable transformation logic on top of normalized records and converted domain fields into analytical features with clear downstream value.

That is why this stage should be understood as more than a collection of ratios. It was the main feature-design layer of the pipeline.

In [ ]:
def compute_valuation(snapshot, current_price: float) -> Dict[str, Optional[float]]:
    net_income = getattr(snapshot, "net_income", None)
    total_equity = getattr(snapshot, "total_equity", None)

    shares = _get_share_count(snapshot)
    fcf = _compute_fcf(snapshot)

    eps = _safe_div(net_income, shares)
    bvps = _safe_div(total_equity, shares)
    fcf_per_share = _safe_div(fcf, shares)

    pe = _safe_div(current_price, eps)
    pb = _safe_div(current_price, bvps)
    p_fcf = _safe_div(current_price, fcf_per_share)

    earnings_yield = _safe_div(eps, current_price)
    fcf_yield = _safe_div(fcf_per_share, current_price)

    return {
        "CurrentPrice": current_price,
        "EPS": eps,
        "BookValuePerShare": bvps,
        "FCFPerShare": fcf_per_share,
        "P/E": pe,
        "P/B": pb,
        "P/FCF": p_fcf,
        "EarningsYield": earnings_yield,
        "FCFYield": fcf_yield,
    }

## Interpretability and Validation Perspective

The feature layer was designed not only for computation, but also for interpretability.

The workflow did not attempt to generate as many outputs as possible. Instead, the focus was on constructing a set of variables that remained explainable, category-based, and directly linked to recognizable analytical questions. This was important because the same engineered variables were later reused in application tables, rule-based summaries, and AI-assisted interpretation.

At the same time, feature engineering did not eliminate the need for validation. A well-defined metric can still become misleading if the underlying annual records are incomplete, inconsistently aligned, or semantically ambiguous. For that reason, this layer improved analytical usefulness, but did not remove the need for interpretation discipline.

## Transition

Once the analytical feature layer was established, the next step was to surface the engineered outputs in a form that supported structured review.

The following chapter explains how the workflow delivered snapshots, feature tables, rule-based summaries, and AI-assisted interpretation through an ordered application interface.